# Advanced Problems with Solutions: Zipping Iterables in Python

This notebook contains advanced examples and practice problems for:

- `zip`
- `itertools.zip_longest`
- lazy iteration
- iterator exhaustion
- finite and infinite iterables
- unzipping / transposition
- chunking with `zip`
- strict length validation
- data alignment
- matrix and tabular transformations
- real-world pipelines using zipped iterables

Each problem includes:
1. a clear task,
2. a solution,
3. tests using `assert`,
4. notes about pitfalls and best practices.

## 0. Setup

In [1]:
from itertools import zip_longest, count, islice, repeat
from collections import deque
from typing import Iterable, Iterator, Any, Callable

## 1. Core Review: `zip` Stops at the Shortest Iterable

`zip(a, b, c)` pulls one value from each iterable at a time and returns tuples.

Important behavior:

- It is lazy.
- It returns an iterator.
- It stops as soon as the shortest iterable is exhausted.

In [2]:
l1 = [1, 2, 3, 4, 5]
l2 = [10, 20, 30]
l3 = ["a", "b", "c", "d"]

result = list(zip(l1, l2, l3))
result

[(1, 10, 'a'), (2, 20, 'b'), (3, 30, 'c')]

In [3]:
assert result == [(1, 10, "a"), (2, 20, "b"), (3, 30, "c")]

### Example: `zip` is lazy

In [4]:
def noisy_numbers(n):
    for i in range(n):
        print(f"producing {i}")
        yield i

z = zip(noisy_numbers(3), ["A", "B", "C"])

print("Nothing has been consumed yet.")
first = next(z)
print("First tuple:", first)
second = next(z)
print("Second tuple:", second)

Nothing has been consumed yet.
producing 0
First tuple: (0, 'A')
producing 1
Second tuple: (1, 'B')


## 2. Problem 1 — Pair Names with Scores

### Task

Given a list of student names and a list of scores, create a dictionary mapping each student to their score.

Use `zip`.

### Requirements

- Ignore extra names or scores automatically.
- Return a dictionary.

In [5]:
names = ["Ada", "Grace", "Linus", "Guido"]
scores = [95, 98, 87]

# Expected:
# {"Ada": 95, "Grace": 98, "Linus": 87}

### Solution 1

In [6]:
student_scores = dict(zip(names, scores))
student_scores

{'Ada': 95, 'Grace': 98, 'Linus': 87}

In [7]:
assert student_scores == {"Ada": 95, "Grace": 98, "Linus": 87}
assert "Guido" not in student_scores

### Best Practice Note

`dict(zip(keys, values))` is clean and common.

However, it silently drops extra values from the longer iterable. If length mismatch is a bug, use `zip(..., strict=True)` in Python 3.10+ or write a validation helper.

## 3. Problem 2 — Detect Length Mismatches with Strict Zipping

### Task

Write a function `strict_pairs(left, right)` that pairs two iterables but raises a `ValueError` if their lengths differ.

This is useful when mismatched lengths indicate corrupted data.

### Requirements

- Return a list of pairs.
- Raise `ValueError` if lengths are different.
- Do not rely on `len()` because the inputs may be generators.

### Solution 2A — Python 3.10+ built-in `strict=True`

In [8]:
def strict_pairs_builtin(left, right):
    try:
        return list(zip(left, right, strict=True))
    except TypeError as exc:
        # This fallback is only for old Python versions that do not support strict=True.
        if "strict" not in str(exc):
            raise
        return strict_pairs_custom(left, right)

### Solution 2B — Custom implementation that works with generators

In [9]:
_SENTINEL = object()

def strict_pairs_custom(left, right):
    result = []
    for a, b in zip_longest(left, right, fillvalue=_SENTINEL):
        if a is _SENTINEL or b is _SENTINEL:
            raise ValueError("Iterables have different lengths")
        result.append((a, b))
    return result

In [10]:
assert strict_pairs_custom([1, 2, 3], ["a", "b", "c"]) == [(1, "a"), (2, "b"), (3, "c")]

try:
    strict_pairs_custom([1, 2, 3], ["a", "b"])
except ValueError as exc:
    assert str(exc) == "Iterables have different lengths"
else:
    raise AssertionError("Expected ValueError")

### Best Practice Note

Avoid using `None` as a filler when detecting missing values. `None` may be a valid data value.

Use a unique sentinel object:

```python
_SENTINEL = object()
```

## 4. Problem 3 — Use `zip_longest` to Align Uneven Columns

### Task

You receive uneven columns from a data export.

Create rows by aligning the columns with `zip_longest`.

### Requirements

- Fill missing values with `"MISSING"`.
- Keep all rows up to the longest column.

In [11]:
ids = [101, 102, 103, 104]
names = ["Ada", "Grace"]
countries = ["UK", "USA", "Finland"]

# Expected:
# [
#   (101, "Ada", "UK"),
#   (102, "Grace", "USA"),
#   (103, "MISSING", "Finland"),
#   (104, "MISSING", "MISSING")
# ]

### Solution 3

In [12]:
aligned_rows = list(zip_longest(ids, names, countries, fillvalue="MISSING"))
aligned_rows

[(101, 'Ada', 'UK'),
 (102, 'Grace', 'USA'),
 (103, 'MISSING', 'Finland'),
 (104, 'MISSING', 'MISSING')]

In [13]:
assert aligned_rows == [
    (101, "Ada", "UK"),
    (102, "Grace", "USA"),
    (103, "MISSING", "Finland"),
    (104, "MISSING", "MISSING"),
]

### Pitfall

`zip_longest` can run forever if any input is infinite and no other input bounds it.

Use `islice` when zipping with infinite iterables.

In [14]:
safe_preview = list(islice(zip_longest(count(1), ["A", "B"], fillvalue="N/A"), 5))
safe_preview

[(1, 'A'), (2, 'B'), (3, 'N/A'), (4, 'N/A'), (5, 'N/A')]

In [15]:
assert safe_preview == [(1, "A"), (2, "B"), (3, "N/A"), (4, "N/A"), (5, "N/A")]

## 5. Problem 4 — Number Records with an Infinite Counter

### Task

Given a finite iterable of events, attach row numbers starting at 1.

### Requirements

- Use `itertools.count`.
- Do not build an intermediate list before numbering.
- Return a list of `(row_number, event)` tuples.

In [16]:
events = ["login", "click", "purchase"]

### Solution 4

In [17]:
numbered_events = list(zip(count(1), events))
numbered_events

[(1, 'login'), (2, 'click'), (3, 'purchase')]

In [18]:
assert numbered_events == [(1, "login"), (2, "click"), (3, "purchase")]

### Best Practice Note

`zip(count(1), finite_iterable)` is safe because normal `zip` stops when the finite iterable ends.

## 6. Problem 5 — Transpose a Matrix with `zip`

### Task

Transpose a rectangular matrix.

Rows become columns.

### Requirements

- Use `zip(*matrix)`.
- Return a list of tuples.

In [19]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

# Expected:
# [(1, 4, 7), (2, 5, 8), (3, 6, 9)]

### Solution 5

In [20]:
transposed = list(zip(*matrix))
transposed

[(1, 4, 7), (2, 5, 8), (3, 6, 9)]

In [21]:
assert transposed == [(1, 4, 7), (2, 5, 8), (3, 6, 9)]

### Advanced Note: Ragged Matrices

`zip(*matrix)` truncates to the shortest row.
Use `zip_longest(*matrix, fillvalue=...)` if rows may have different lengths.

In [22]:
ragged = [
    [1, 2, 3],
    [4, 5],
    [6, 7, 8, 9],
]

ragged_transposed = list(zip_longest(*ragged, fillvalue=None))
ragged_transposed

[(1, 4, 6), (2, 5, 7), (3, None, 8), (None, None, 9)]

In [23]:
assert ragged_transposed == [
    (1, 4, 6),
    (2, 5, 7),
    (3, None, 8),
    (None, None, 9),
]

## 7. Problem 6 — Compute Elementwise Vector Operations

### Task

Given two numeric vectors, compute:

1. elementwise sums,
2. elementwise differences,
3. dot product.

### Requirements

- Use `zip`.
- Validate that both vectors have the same length.
- Use generator expressions where appropriate.

In [24]:
v1 = [3, 4, 5]
v2 = [10, 20, 30]

### Solution 6

In [25]:
def same_length_zip(left, right):
    sentinel = object()
    for a, b in zip_longest(left, right, fillvalue=sentinel):
        if a is sentinel or b is sentinel:
            raise ValueError("Vectors must have the same length")
        yield a, b

def vector_sum(left, right):
    return [a + b for a, b in same_length_zip(left, right)]

def vector_difference(left, right):
    return [a - b for a, b in same_length_zip(left, right)]

def dot_product(left, right):
    return sum(a * b for a, b in same_length_zip(left, right))

sum_result = vector_sum(v1, v2)
diff_result = vector_difference(v1, v2)
dot_result = dot_product(v1, v2)

sum_result, diff_result, dot_result

([13, 24, 35], [-7, -16, -25], 260)

In [26]:
assert sum_result == [13, 24, 35]
assert diff_result == [-7, -16, -25]
assert dot_result == 260

try:
    dot_product([1, 2], [10, 20, 30])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for mismatched vector lengths")

## 8. Problem 7 — Chunk a Sequence into Fixed-Size Groups

### Task

Write a function `chunk_exact(iterable, size)` that groups values into chunks of exactly `size`.

### Requirements

- Use the classic `zip(*[iter(iterable)] * size)` technique.
- Drop incomplete final chunks.
- Return an iterator.

In [27]:
data = list(range(10))

# Expected for size=3:
# [(0, 1, 2), (3, 4, 5), (6, 7, 8)]

### Solution 7

In [28]:
def chunk_exact(iterable, size):
    if size <= 0:
        raise ValueError("size must be positive")
    it = iter(iterable)
    return zip(*[it] * size)

chunks = list(chunk_exact(data, 3))
chunks

[(0, 1, 2), (3, 4, 5), (6, 7, 8)]

In [29]:
assert chunks == [(0, 1, 2), (3, 4, 5), (6, 7, 8)]

try:
    list(chunk_exact([1, 2, 3], 0))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for size <= 0")

### Why This Works

`[it] * size` creates multiple references to the same iterator.

Then `zip` asks each reference for one item in order, producing fixed-size groups.

## 9. Problem 8 — Chunk a Sequence and Keep the Remainder

### Task

Write `chunk_all(iterable, size, fillvalue=None)`.

Unlike `chunk_exact`, this function should keep the final incomplete chunk using `zip_longest`.

### Requirements

- Use `zip_longest`.
- Fill missing values with `fillvalue`.
- Return an iterator.

### Solution 8

In [30]:
def chunk_all(iterable, size, fillvalue=None):
    if size <= 0:
        raise ValueError("size must be positive")
    it = iter(iterable)
    return zip_longest(*[it] * size, fillvalue=fillvalue)

all_chunks = list(chunk_all(range(10), 3, fillvalue="PAD"))
all_chunks

[(0, 1, 2), (3, 4, 5), (6, 7, 8), (9, 'PAD', 'PAD')]

In [31]:
assert all_chunks == [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    (9, "PAD", "PAD"),
]

### Best Practice Warning

If `fillvalue=None`, you may not be able to tell whether `None` was original data or padding.

Use a custom sentinel object when this distinction matters.

## 10. Problem 9 — Sliding Adjacent Pairs

### Task

Given a sequence, create adjacent pairs:

```python
[10, 20, 30, 40]
```

should become:

```python
[(10, 20), (20, 30), (30, 40)]
```

### Requirements

- Use `zip`.
- Do not use indexes.

### Solution 9A — For sequences

In [32]:
values = [10, 20, 30, 40]

adjacent_pairs = list(zip(values, values[1:]))
adjacent_pairs

[(10, 20), (20, 30), (30, 40)]

In [33]:
assert adjacent_pairs == [(10, 20), (20, 30), (30, 40)]

### Solution 9B — For any iterable using `tee`

In [34]:
from itertools import tee

def pairwise_custom(iterable):
    first, second = tee(iterable)
    next(second, None)
    return zip(first, second)

assert list(pairwise_custom([10, 20, 30, 40])) == [(10, 20), (20, 30), (30, 40)]
assert list(pairwise_custom(iter([1, 2, 3]))) == [(1, 2), (2, 3)]

### Best Practice Note

For Python 3.10+, `itertools.pairwise` is available and should usually be preferred.

In [35]:
try:
    from itertools import pairwise
    assert list(pairwise([10, 20, 30, 40])) == [(10, 20), (20, 30), (30, 40)]
except ImportError:
    pass

## 11. Problem 10 — Calculate Differences Between Consecutive Measurements

### Task

Given timestamped measurements, calculate the difference between consecutive values.

### Input

```python
measurements = [
    ("09:00", 100),
    ("10:00", 108),
    ("11:00", 105),
    ("12:00", 111),
]
```

### Expected Output

```python
[
    ("09:00 -> 10:00", 8),
    ("10:00 -> 11:00", -3),
    ("11:00 -> 12:00", 6),
]
```

### Requirements

- Use adjacent pairs.
- Return a list.

In [36]:
measurements = [
    ("09:00", 100),
    ("10:00", 108),
    ("11:00", 105),
    ("12:00", 111),
]

### Solution 10

In [37]:
def consecutive_differences(records):
    return [
        (f"{t1} -> {t2}", value2 - value1)
        for (t1, value1), (t2, value2) in zip(records, records[1:])
    ]

diffs = consecutive_differences(measurements)
diffs

[('09:00 -> 10:00', 8), ('10:00 -> 11:00', -3), ('11:00 -> 12:00', 6)]

In [38]:
assert diffs == [
    ("09:00 -> 10:00", 8),
    ("10:00 -> 11:00", -3),
    ("11:00 -> 12:00", 6),
]

## 12. Problem 11 — Merge Headers with Rows

### Task

Convert rows of data into dictionaries using a header row.

### Input

```python
headers = ["id", "name", "score"]
rows = [
    [1, "Ada", 95],
    [2, "Grace", 98],
]
```

### Expected Output

```python
[
    {"id": 1, "name": "Ada", "score": 95},
    {"id": 2, "name": "Grace", "score": 98},
]
```

### Requirements

- Use `zip`.
- Validate that each row has the same number of fields as the header.

In [39]:
headers = ["id", "name", "score"]
rows = [
    [1, "Ada", 95],
    [2, "Grace", 98],
]

### Solution 11

In [40]:
def rows_to_dicts(headers, rows):
    result = []
    for row_number, row in zip(count(1), rows):
        if len(row) != len(headers):
            raise ValueError(
                f"Row {row_number} has {len(row)} fields; expected {len(headers)}"
            )
        result.append(dict(zip(headers, row)))
    return result

records = rows_to_dicts(headers, rows)
records

[{'id': 1, 'name': 'Ada', 'score': 95},
 {'id': 2, 'name': 'Grace', 'score': 98}]

In [41]:
assert records == [
    {"id": 1, "name": "Ada", "score": 95},
    {"id": 2, "name": "Grace", "score": 98},
]

try:
    rows_to_dicts(headers, [[1, "Ada"]])
except ValueError as exc:
    assert "Row 1" in str(exc)
else:
    raise AssertionError("Expected ValueError")

## 13. Problem 12 — Align Two Event Streams by Position

### Task

You have expected events and actual events. Compare them position by position.

### Requirements

Return a list of dictionaries with:

- `position`
- `expected`
- `actual`
- `status`

Status rules:

- `"match"` when expected equals actual
- `"mismatch"` when both exist but differ
- `"missing_actual"` when actual is missing
- `"unexpected_actual"` when expected is missing

In [42]:
expected_events = ["start", "load", "validate", "save", "end"]
actual_events = ["start", "load", "save", "end", "cleanup"]

### Solution 12

In [43]:
MISSING = object()

def compare_streams(expected, actual):
    comparisons = []
    for position, (exp, act) in zip(
        count(1),
        zip_longest(expected, actual, fillvalue=MISSING)
    ):
        if exp is MISSING:
            status = "unexpected_actual"
            exp_value = None
        else:
            exp_value = exp

        if act is MISSING:
            status = "missing_actual"
            act_value = None
        else:
            act_value = act

        if exp is not MISSING and act is not MISSING:
            status = "match" if exp == act else "mismatch"

        comparisons.append({
            "position": position,
            "expected": exp_value,
            "actual": act_value,
            "status": status,
        })

    return comparisons

comparison_result = compare_streams(expected_events, actual_events)
comparison_result

[{'position': 1, 'expected': 'start', 'actual': 'start', 'status': 'match'},
 {'position': 2, 'expected': 'load', 'actual': 'load', 'status': 'match'},
 {'position': 3,
  'expected': 'validate',
  'actual': 'save',
  'status': 'mismatch'},
 {'position': 4, 'expected': 'save', 'actual': 'end', 'status': 'mismatch'},
 {'position': 5, 'expected': 'end', 'actual': 'cleanup', 'status': 'mismatch'}]

In [44]:
assert comparison_result == [
    {"position": 1, "expected": "start", "actual": "start", "status": "match"},
    {"position": 2, "expected": "load", "actual": "load", "status": "match"},
    {"position": 3, "expected": "validate", "actual": "save", "status": "mismatch"},
    {"position": 4, "expected": "save", "actual": "end", "status": "mismatch"},
    {"position": 5, "expected": "end", "actual": "cleanup", "status": "mismatch"},
]

### Extra Example: Missing and Unexpected Events

In [45]:
assert compare_streams(["a", "b", "c"], ["a"]) == [
    {"position": 1, "expected": "a", "actual": "a", "status": "match"},
    {"position": 2, "expected": "b", "actual": None, "status": "missing_actual"},
    {"position": 3, "expected": "c", "actual": None, "status": "missing_actual"},
]

assert compare_streams(["a"], ["a", "b"]) == [
    {"position": 1, "expected": "a", "actual": "a", "status": "match"},
    {"position": 2, "expected": None, "actual": "b", "status": "unexpected_actual"},
]

## 14. Problem 13 — Combine Multiple Feature Columns into Records

### Task

You are given several columns from a machine-learning feature table.

Create a list of feature dictionaries.

### Requirements

- Use `zip`.
- Create dictionaries with keys:
  - `"age"`
  - `"income"`
  - `"clicked"`
- Validate equal lengths.

In [46]:
ages = [25, 40, 31]
incomes = [50_000, 90_000, 72_000]
clicked = [True, False, True]

### Solution 13

In [47]:
def build_feature_records(ages, incomes, clicked):
    records = []
    sentinel = object()

    for age, income, did_click in zip_longest(
        ages, incomes, clicked, fillvalue=sentinel
    ):
        if sentinel in (age, income, did_click):
            raise ValueError("Feature columns must have equal lengths")

        records.append({
            "age": age,
            "income": income,
            "clicked": did_click,
        })

    return records

feature_records = build_feature_records(ages, incomes, clicked)
feature_records

[{'age': 25, 'income': 50000, 'clicked': True},
 {'age': 40, 'income': 90000, 'clicked': False},
 {'age': 31, 'income': 72000, 'clicked': True}]

In [48]:
assert feature_records == [
    {"age": 25, "income": 50_000, "clicked": True},
    {"age": 40, "income": 90_000, "clicked": False},
    {"age": 31, "income": 72_000, "clicked": True},
]

try:
    build_feature_records([1, 2], [100], [True, False])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 15. Problem 14 — Weighted Average Using `zip`

### Task

Calculate a weighted average.

### Formula

```python
weighted_average = sum(value * weight) / sum(weights)
```

### Requirements

- Use `zip`.
- Validate equal lengths.
- Raise `ValueError` if total weight is zero.

In [49]:
values = [80, 90, 100]
weights = [0.2, 0.3, 0.5]

### Solution 14

In [50]:
def weighted_average(values, weights):
    pairs = list(same_length_zip(values, weights))
    total_weight = sum(weight for _, weight in pairs)

    if total_weight == 0:
        raise ValueError("Total weight cannot be zero")

    return sum(value * weight for value, weight in pairs) / total_weight

avg = weighted_average(values, weights)
avg

93.0

In [51]:
assert avg == 93.0

try:
    weighted_average([10, 20], [0, 0])
except ValueError as exc:
    assert "zero" in str(exc)
else:
    raise AssertionError("Expected ValueError for zero total weight")

## 16. Problem 15 — Interleave Two Iterables

### Task

Write `interleave(left, right)`.

Example:

```python
interleave(["A", "B", "C"], [1, 2, 3])
```

should produce:

```python
["A", 1, "B", 2, "C", 3]
```

### Requirements

- Use `zip`.
- Validate that lengths are equal.
- Return a list.

### Solution 15

In [52]:
def interleave(left, right):
    output = []
    for a, b in same_length_zip(left, right):
        output.extend([a, b])
    return output

interleaved = interleave(["A", "B", "C"], [1, 2, 3])
interleaved

['A', 1, 'B', 2, 'C', 3]

In [53]:
assert interleaved == ["A", 1, "B", 2, "C", 3]

try:
    interleave(["A", "B"], [1])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 17. Problem 16 — Pairwise Validation of Sorted Data

### Task

Write a function `is_non_decreasing(iterable)`.

It should return `True` when every item is less than or equal to the next item.

### Requirements

- Work with any iterable.
- Use pairwise zipping.
- Return `True` for empty or single-item iterables.

### Solution 16

In [54]:
def is_non_decreasing(iterable):
    first, second = tee(iterable)
    next(second, None)
    return all(a <= b for a, b in zip(first, second))

assert is_non_decreasing([])
assert is_non_decreasing([10])
assert is_non_decreasing([1, 2, 2, 3])
assert not is_non_decreasing([1, 3, 2])
assert is_non_decreasing(iter([1, 2, 3]))

## 18. Problem 17 — Compare Two Dictionaries by Key Order

### Task

Given two dictionaries with the same insertion order of keys, produce a comparison table.

### Input

```python
before = {"cpu": 40, "memory": 70, "disk": 55}
after = {"cpu": 45, "memory": 68, "disk": 80}
```

### Expected Output

```python
[
    ("cpu", 40, 45, 5),
    ("memory", 70, 68, -2),
    ("disk", 55, 80, 25),
]
```

### Requirements

- Use `zip`.
- Validate that keys are identical and in the same order.

In [55]:
before = {"cpu": 40, "memory": 70, "disk": 55}
after = {"cpu": 45, "memory": 68, "disk": 80}

### Solution 17

In [56]:
def compare_ordered_dict_values(before, after):
    if list(before.keys()) != list(after.keys()):
        raise ValueError("Dictionaries must have the same keys in the same order")

    return [
        (key_before, value_before, value_after, value_after - value_before)
        for (key_before, value_before), (key_after, value_after)
        in zip(before.items(), after.items())
    ]

dict_comparison = compare_ordered_dict_values(before, after)
dict_comparison

[('cpu', 40, 45, 5), ('memory', 70, 68, -2), ('disk', 55, 80, 25)]

In [57]:
assert dict_comparison == [
    ("cpu", 40, 45, 5),
    ("memory", 70, 68, -2),
    ("disk", 55, 80, 25),
]

try:
    compare_ordered_dict_values({"a": 1, "b": 2}, {"b": 2, "a": 1})
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 19. Problem 18 — Build a Mini CSV Parser

### Task

Write a tiny CSV-style parser for already-split rows.

### Input

```python
rows = [
    ["id", "name", "score"],
    ["1", "Ada", "95"],
    ["2", "Grace", "98"],
]
```

### Expected Output

```python
[
    {"id": "1", "name": "Ada", "score": "95"},
    {"id": "2", "name": "Grace", "score": "98"},
]
```

### Requirements

- Use the first row as headers.
- Use `zip` to combine headers and fields.
- Validate row lengths.

In [58]:
split_csv_rows = [
    ["id", "name", "score"],
    ["1", "Ada", "95"],
    ["2", "Grace", "98"],
]

### Solution 18

In [59]:
def parse_split_csv(rows):
    rows = iter(rows)

    try:
        headers = next(rows)
    except StopIteration:
        return []

    parsed = []
    for line_number, row in zip(count(2), rows):
        if len(row) != len(headers):
            raise ValueError(
                f"Line {line_number}: expected {len(headers)} fields, got {len(row)}"
            )
        parsed.append(dict(zip(headers, row)))

    return parsed

parsed_csv = parse_split_csv(split_csv_rows)
parsed_csv

[{'id': '1', 'name': 'Ada', 'score': '95'},
 {'id': '2', 'name': 'Grace', 'score': '98'}]

In [60]:
assert parsed_csv == [
    {"id": "1", "name": "Ada", "score": "95"},
    {"id": "2", "name": "Grace", "score": "98"},
]

assert parse_split_csv([]) == []

try:
    parse_split_csv([["id", "name"], ["1"]])
except ValueError as exc:
    assert "Line 2" in str(exc)
else:
    raise AssertionError("Expected ValueError")

## 20. Problem 19 — Safe Preview of `zip_longest` with Infinite Data

### Task

Create a function `preview_zip_longest(limit, *iterables, fillvalue=None)`.

### Requirements

- Use `zip_longest`.
- Use `islice` to limit the output.
- Return a list.
- Raise `ValueError` if `limit` is negative.

### Solution 19

In [61]:
def preview_zip_longest(limit, *iterables, fillvalue=None):
    if limit < 0:
        raise ValueError("limit must be non-negative")

    return list(islice(zip_longest(*iterables, fillvalue=fillvalue), limit))

preview = preview_zip_longest(7, count(1), ["A", "B", "C"], fillvalue="N/A")
preview

[(1, 'A'), (2, 'B'), (3, 'C'), (4, 'N/A'), (5, 'N/A'), (6, 'N/A'), (7, 'N/A')]

In [62]:
assert preview == [
    (1, "A"),
    (2, "B"),
    (3, "C"),
    (4, "N/A"),
    (5, "N/A"),
    (6, "N/A"),
    (7, "N/A"),
]

try:
    preview_zip_longest(-1, [1, 2, 3])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 21. Problem 20 — Rolling Windows with `zip`

### Task

Write `windows(sequence, size)` that returns overlapping windows.

Example:

```python
windows([1, 2, 3, 4, 5], 3)
```

should return:

```python
[(1, 2, 3), (2, 3, 4), (3, 4, 5)]
```

### Requirements

- Use `zip`.
- Work for sequences.
- Raise `ValueError` if `size <= 0`.

### Solution 20

In [63]:
def windows(sequence, size):
    if size <= 0:
        raise ValueError("size must be positive")
    return list(zip(*(sequence[i:] for i in range(size))))

window_result = windows([1, 2, 3, 4, 5], 3)
window_result

[(1, 2, 3), (2, 3, 4), (3, 4, 5)]

In [64]:
assert window_result == [(1, 2, 3), (2, 3, 4), (3, 4, 5)]
assert windows([1, 2], 3) == []

try:
    windows([1, 2, 3], 0)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 22. Problem 21 — Find Consecutive Runs

### Task

Given a list of integers, find ranges of consecutive increasing-by-1 runs.

Example:

```python
[1, 2, 3, 7, 8, 10]
```

should produce:

```python
[(1, 3), (7, 8), (10, 10)]
```

### Requirements

- Use adjacent pairs.
- Return a list of `(start, end)` tuples.

### Solution 21

In [65]:
def consecutive_runs(values):
    values = list(values)
    if not values:
        return []

    runs = []
    start = values[0]

    for current_value, next_value in zip(values, values[1:]):
        if next_value != current_value + 1:
            runs.append((start, current_value))
            start = next_value

    runs.append((start, values[-1]))
    return runs

runs_result = consecutive_runs([1, 2, 3, 7, 8, 10])
runs_result

[(1, 3), (7, 8), (10, 10)]

In [66]:
assert runs_result == [(1, 3), (7, 8), (10, 10)]
assert consecutive_runs([]) == []
assert consecutive_runs([5]) == [(5, 5)]
assert consecutive_runs([1, 3, 5]) == [(1, 1), (3, 3), (5, 5)]

## 23. Problem 22 — Column-Wise Aggregation

### Task

Given a rectangular table of numbers, compute the sum of each column.

### Requirements

- Use `zip(*table)`.
- Validate that the table is rectangular.
- Return a list.

In [67]:
table = [
    [10, 20, 30],
    [1, 2, 3],
    [100, 200, 300],
]

### Solution 22

In [68]:
def column_sums(table):
    if not table:
        return []

    expected_length = len(table[0])
    for row_number, row in zip(count(1), table):
        if len(row) != expected_length:
            raise ValueError(f"Row {row_number} has the wrong length")

    return [sum(column) for column in zip(*table)]

col_sums = column_sums(table)
col_sums

[111, 222, 333]

In [69]:
assert col_sums == [111, 222, 333]
assert column_sums([]) == []

try:
    column_sums([[1, 2], [3]])
except ValueError as exc:
    assert "Row 2" in str(exc)
else:
    raise AssertionError("Expected ValueError")

## 24. Problem 23 — Matrix Addition

### Task

Add two matrices of the same shape.

### Requirements

- Use nested `zip`.
- Validate identical shape.
- Return a list of lists.

In [70]:
A = [
    [1, 2, 3],
    [4, 5, 6],
]

B = [
    [10, 20, 30],
    [40, 50, 60],
]

### Solution 23

In [71]:
def matrix_shape(matrix):
    if not matrix:
        return (0, 0)

    width = len(matrix[0])
    for row_number, row in zip(count(1), matrix):
        if len(row) != width:
            raise ValueError(f"Matrix is ragged at row {row_number}")

    return (len(matrix), width)

def add_matrices(left, right):
    if matrix_shape(left) != matrix_shape(right):
        raise ValueError("Matrices must have the same shape")

    return [
        [a + b for a, b in zip(row_left, row_right)]
        for row_left, row_right in zip(left, right)
    ]

matrix_sum = add_matrices(A, B)
matrix_sum

[[11, 22, 33], [44, 55, 66]]

In [72]:
assert matrix_sum == [
    [11, 22, 33],
    [44, 55, 66],
]

try:
    add_matrices([[1, 2]], [[1, 2], [3, 4]])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 25. Problem 24 — Matrix Multiplication Using `zip`

### Task

Multiply two matrices.

### Requirements

- Use `zip(*right)` to access columns of the right matrix.
- Validate dimensions.
- Return a list of lists.

In [73]:
left_matrix = [
    [1, 2, 3],
    [4, 5, 6],
]

right_matrix = [
    [7, 8],
    [9, 10],
    [11, 12],
]

# Expected:
# [
#   [58, 64],
#   [139, 154],
# ]

### Solution 24

In [74]:
def matmul(left, right):
    left_rows, left_cols = matrix_shape(left)
    right_rows, right_cols = matrix_shape(right)

    if left_cols != right_rows:
        raise ValueError(
            f"Cannot multiply shapes {(left_rows, left_cols)} and {(right_rows, right_cols)}"
        )

    right_columns = list(zip(*right))

    return [
        [
            sum(a * b for a, b in zip(left_row, right_col))
            for right_col in right_columns
        ]
        for left_row in left
    ]

matmul_result = matmul(left_matrix, right_matrix)
matmul_result

[[58, 64], [139, 154]]

In [75]:
assert matmul_result == [
    [58, 64],
    [139, 154],
]

try:
    matmul([[1, 2, 3]], [[1, 2]])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 26. Problem 25 — Group Fields by Record Using `zip`

### Task

You have three equal-length lists:

```python
products = ["book", "pen", "bag"]
prices = [12.5, 1.2, 35.0]
quantities = [2, 10, 1]
```

Build invoice lines with subtotal.

### Expected Output

```python
[
    {"product": "book", "price": 12.5, "quantity": 2, "subtotal": 25.0},
    {"product": "pen", "price": 1.2, "quantity": 10, "subtotal": 12.0},
    {"product": "bag", "price": 35.0, "quantity": 1, "subtotal": 35.0},
]
```

In [76]:
products = ["book", "pen", "bag"]
prices = [12.5, 1.2, 35.0]
quantities = [2, 10, 1]

### Solution 25

In [77]:
def invoice_lines(products, prices, quantities):
    lines = []
    sentinel = object()

    for product, price, quantity in zip_longest(
        products, prices, quantities, fillvalue=sentinel
    ):
        if sentinel in (product, price, quantity):
            raise ValueError("All invoice columns must have equal lengths")

        lines.append({
            "product": product,
            "price": price,
            "quantity": quantity,
            "subtotal": price * quantity,
        })

    return lines

invoice = invoice_lines(products, prices, quantities)
invoice

[{'product': 'book', 'price': 12.5, 'quantity': 2, 'subtotal': 25.0},
 {'product': 'pen', 'price': 1.2, 'quantity': 10, 'subtotal': 12.0},
 {'product': 'bag', 'price': 35.0, 'quantity': 1, 'subtotal': 35.0}]

In [78]:
assert invoice == [
    {"product": "book", "price": 12.5, "quantity": 2, "subtotal": 25.0},
    {"product": "pen", "price": 1.2, "quantity": 10, "subtotal": 12.0},
    {"product": "bag", "price": 35.0, "quantity": 1, "subtotal": 35.0},
]

## 27. Problem 26 — Create a `zip_with` Helper

### Task

Implement a function similar to functional programming's `zipWith`.

`zip_with(fn, left, right)` should apply `fn(a, b)` to each pair.

### Example

```python
zip_with(lambda a, b: a + b, [1, 2], [10, 20])
```

returns:

```python
[11, 22]
```

### Requirements

- Use `zip`.
- Stop at the shortest iterable.
- Return a list.

### Solution 26

In [79]:
def zip_with(fn, left, right):
    return [fn(a, b) for a, b in zip(left, right)]

assert zip_with(lambda a, b: a + b, [1, 2], [10, 20]) == [11, 22]
assert zip_with(lambda a, b: a * b, [1, 2, 3], [10, 20]) == [10, 40]

### Strict Version

In [80]:
def strict_zip_with(fn, left, right):
    return [fn(a, b) for a, b in same_length_zip(left, right)]

assert strict_zip_with(lambda a, b: a + b, [1, 2], [10, 20]) == [11, 22]

try:
    strict_zip_with(lambda a, b: a + b, [1, 2, 3], [10, 20])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 28. Problem 27 — Side-by-Side Pretty Table

### Task

Print two columns side by side.

### Input

```python
left = ["Python", "JavaScript", "Rust"]
right = ["easy to read", "web native", "memory safe"]
```

### Expected printed format

```text
Python      | easy to read
JavaScript  | web native
Rust        | memory safe
```

### Requirements

- Use `zip`.
- Align the first column.

### Solution 27

In [81]:
left = ["Python", "JavaScript", "Rust"]
right = ["easy to read", "web native", "memory safe"]

def side_by_side(left, right):
    width = max(map(len, left), default=0)
    return "\n".join(
        f"{left_item:<{width}} | {right_item}"
        for left_item, right_item in zip(left, right)
    )

table_text = side_by_side(left, right)
print(table_text)

Python     | easy to read
JavaScript | web native
Rust       | memory safe


In [82]:
assert table_text == (
    "Python     | easy to read\n"
    "JavaScript | web native\n"
    "Rust       | memory safe"
)

## 29. Problem 28 — Reconstruct a Dictionary from Separate Key and Value Streams

### Task

Given key and value generators, create a dictionary.

### Requirements

- Use `zip`.
- Accept any iterable.
- Stop at the shortest iterable.

### Solution 28

In [83]:
def key_stream():
    yield "a"
    yield "b"
    yield "c"

def value_stream():
    yield 10
    yield 20

stream_dict = dict(zip(key_stream(), value_stream()))
stream_dict

{'a': 10, 'b': 20}

In [84]:
assert stream_dict == {"a": 10, "b": 20}

### Strict Alternative

In [85]:
def strict_dict(keys, values):
    return dict(strict_pairs_custom(keys, values))

assert strict_dict(["a", "b"], [1, 2]) == {"a": 1, "b": 2}

try:
    strict_dict(["a", "b", "c"], [1, 2])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

## 30. Problem 29 — Mark Missing Survey Answers

### Task

You have survey questions and answers. Some answers may be missing.

### Requirements

- Use `zip_longest`.
- Fill missing answers with `"NO ANSWER"`.
- Return a dictionary mapping question to answer.
- Ignore unexpected extra answers by storing them under generated keys like `"EXTRA_1"`.

In [86]:
questions = ["name", "age", "country"]
answers = ["Ada", "36"]

### Solution 29

In [87]:
def survey_dict(questions, answers):
    missing_question = object()
    result = {}
    extra_counter = count(1)

    for question, answer in zip_longest(
        questions, answers, fillvalue=missing_question
    ):
        if question is missing_question:
            question = f"EXTRA_{next(extra_counter)}"

        if answer is missing_question:
            answer = "NO ANSWER"

        result[question] = answer

    return result

survey_result = survey_dict(questions, answers)
survey_result

{'name': 'Ada', 'age': '36', 'country': 'NO ANSWER'}

In [88]:
assert survey_result == {
    "name": "Ada",
    "age": "36",
    "country": "NO ANSWER",
}

assert survey_dict(["q1"], ["a1", "a2", "a3"]) == {
    "q1": "a1",
    "EXTRA_1": "a2",
    "EXTRA_2": "a3",
}

## 31. Problem 30 — Diagnose Iterator Exhaustion

### Task

Explain and fix this bug:

```python
z = zip([1, 2, 3], ["a", "b", "c"])
first_pass = list(z)
second_pass = list(z)
```

`second_pass` is empty.

### Requirements

- Show the bug.
- Fix it by recreating the `zip` object.
- Show another fix using `list` to materialize results.

### Solution 30

In [89]:
z = zip([1, 2, 3], ["a", "b", "c"])
first_pass = list(z)
second_pass = list(z)

first_pass, second_pass

([(1, 'a'), (2, 'b'), (3, 'c')], [])

In [90]:
assert first_pass == [(1, "a"), (2, "b"), (3, "c")]
assert second_pass == []

### Fix 1 — Recreate the `zip` object

In [91]:
def make_zip():
    return zip([1, 2, 3], ["a", "b", "c"])

first = list(make_zip())
second = list(make_zip())

assert first == second == [(1, "a"), (2, "b"), (3, "c")]

### Fix 2 — Materialize once if you need reuse

In [92]:
pairs = list(zip([1, 2, 3], ["a", "b", "c"]))

first_reuse = list(pairs)
second_reuse = list(pairs)

assert first_reuse == second_reuse == [(1, "a"), (2, "b"), (3, "c")]

## 32. Challenge Problem — Build a Small Data Alignment Report

### Task

Write `alignment_report(*columns, fillvalue="MISSING")`.

It should accept any number of columns and produce:

1. `rows`: the `zip_longest` aligned rows,
2. `column_lengths`: lengths of each input column if known,
3. `max_width`: the number of rows in the aligned output,
4. `missing_count`: total number of filled missing values.

### Requirements

- Use `zip_longest`.
- Support any number of columns.
- Use a sentinel to count missing positions safely.
- Return a dictionary.

### Solution — Challenge Problem

In [93]:
def alignment_report(*columns, fillvalue="MISSING"):
    sentinel = object()

    rows_with_sentinel = list(zip_longest(*columns, fillvalue=sentinel))

    missing_count = sum(
        value is sentinel
        for row in rows_with_sentinel
        for value in row
    )

    rows = [
        tuple(fillvalue if value is sentinel else value for value in row)
        for row in rows_with_sentinel
    ]

    column_lengths = []
    for column in columns:
        try:
            column_lengths.append(len(column))
        except TypeError:
            column_lengths.append(None)

    return {
        "rows": rows,
        "column_lengths": column_lengths,
        "max_width": len(rows),
        "missing_count": missing_count,
    }

report = alignment_report(
    [1, 2, 3],
    ["a"],
    [True, False],
    fillvalue="N/A",
)

report

{'rows': [(1, 'a', True), (2, 'N/A', False), (3, 'N/A', 'N/A')],
 'column_lengths': [3, 1, 2],
 'max_width': 3,
 'missing_count': 3}

In [94]:
assert report == {
    "rows": [
        (1, "a", True),
        (2, "N/A", False),
        (3, "N/A", "N/A"),
    ],
    "column_lengths": [3, 1, 2],
    "max_width": 3,
    "missing_count": 3,
}

## 33. More Practice Prompts

Try solving these before looking back at the patterns above.

1. Write `strict_zip_many(*iterables)` that validates all inputs have the same length.
2. Write `transpose_ragged(matrix, fillvalue="")`.
3. Write `enumerate_from(iterable, start)` using only `zip` and `count`.
4. Write `zip_dicts_by_key(dict1, dict2)` that pairs values for common keys only.
5. Write `detect_changes(before, after)` that returns changed positions using `zip_longest`.
6. Write `moving_average(values, window_size)` using rolling windows.
7. Write `column_maxes(table)` using `zip(*table)`.
8. Write `pairwise_ratios(values)` using adjacent pairs.
9. Write `merge_three_logs(log1, log2, log3)` using `zip_longest`.
10. Write `format_parallel_lists(*lists)` to print multiple aligned columns.

## 34. Summary: Best Practices

- Use `zip` when all you need is parallel iteration up to the shortest iterable.
- Use `zip_longest` when missing data should be preserved.
- Use a unique sentinel object when missing values must be distinguished from real values.
- Be careful with infinite iterables and `zip_longest`.
- Use `islice` to preview infinite or potentially unbounded zipped data.
- Remember that `zip` returns an iterator and becomes exhausted after use.
- Use `dict(zip(keys, values))` for clean key-value construction.
- Use `zip(*matrix)` to transpose rectangular data.
- Validate lengths when mismatch would indicate a bug.
- Prefer readable helper functions when zipping logic appears repeatedly.